**Task**

Compare the performance of the individual modality
models with the multimodal fusion model.


In [1]:
import os
import glob
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import ttest_ind

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import roc_auc_score

from statsmodels.stats.contingency_tables import mcnemar

from scipy import stats

from google.colab import drive

drive.mount(
    '/content/drive'
)

Mounted at /content/drive


In [2]:
base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

print(
    "Processed path:",
    processed_path
)

print(
    "Model path:",
    model_path
)

Processed path: /content/drive/MyDrive/dissertation_project/data/processed
Model path: /content/drive/MyDrive/dissertation_project/data/models


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [4]:
# Find available metric files
all_csv_files = glob.glob(
    f'{processed_path}/*.csv'
)

print(
    "CSV files found:",
    len(all_csv_files)
)

for file in all_csv_files:
    print(
        os.path.basename(file)
    )

CSV files found: 25
matched_final.csv
lab_feature_dictionary.csv
2_image_reference.csv
1_structured_reference.csv
3_text_reference.csv
structured_processed.csv
text_processed.csv
test_calibration_ids.csv
test_calibration_pool.csv
text_baseline_metrics.csv
text_training_history.csv
text_features_heldout.csv
text_features_development.csv
image_baseline_metrics.csv
image_training_history.csv
image_features_heldout.csv
image_features_development.csv
structured_baseline_metrics.csv
structured_features_mlp_heldout.csv
structured_training_history.csv
structured_features_mlp.csv
fusion_layer1_metrics.csv
fusion_layer1_training_history.csv
fusion_features_layer1.csv
fusion_features_layer1_heldout.csv


In [5]:
# Load metric files
metric_files = [
    file
    for file in all_csv_files
    if (
        'metric' in
        os.path.basename(file).lower()
    )
]

print(
    "Metric files found:"
)

for file in metric_files:
    print(
        os.path.basename(file)
    )

Metric files found:
text_baseline_metrics.csv
image_baseline_metrics.csv
structured_baseline_metrics.csv
fusion_layer1_metrics.csv


In [6]:
# Load multimodal fusion result
fusion_file = (
    f'{processed_path}/'
    'fusion_layer1_metrics.csv'
)

if os.path.exists(fusion_file):

    fusion_metrics = pd.read_csv(
        fusion_file
    )

    print(
        "Fusion metrics loaded:"
    )

    display(
        fusion_metrics
    )

else:

    print(
        "Fusion metrics file not found:"
    )

    print(
        fusion_file
    )

Fusion metrics loaded:


,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.911894,0.819672,0.847458,0.833333,0.918382
1,Support Devices,0.766520,0.623656,0.763158,0.686391,0.869990
2,Pleural Effusion,0.872247,0.736842,0.750000,0.743363,0.938074
3,Lung Opacity,0.947137,0.950820,0.865672,0.906250,0.982183
4,Atelectasis,0.885463,0.796875,0.796875,0.796875,0.950633
5,Cardiomegaly,0.920705,0.930233,0.727273,0.816327,0.946934
6,Edema,0.969163,0.791667,0.904762,0.844444,0.988442


In [7]:
# Calculate fusion averages
fusion_summary = pd.DataFrame({

    'Model': [
        'Multimodal Fusion'
    ],

    'Accuracy': [
        fusion_metrics[
            'accuracy'
        ].mean()
    ],

    'Precision': [
        fusion_metrics[
            'precision'
        ].mean()
    ],

    'Recall': [
        fusion_metrics[
            'recall'
        ].mean()
    ],

    'F1': [
        fusion_metrics[
            'f1'
        ].mean()
    ],

    'ROC-AUC': [
        fusion_metrics[
            'roc_auc'
        ].mean()
    ]
})

display(
    fusion_summary
)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Multimodal Fusion,0.896161,0.807109,0.807885,0.803855,0.942091


In [8]:
# Load previous modality results
candidate_metric_tables = []

for file in metric_files:

    try:

        df = pd.read_csv(
            file
        )

        required_columns = {
            'accuracy',
            'precision',
            'recall',
            'f1',
            'roc_auc'
        }

        if required_columns.issubset(
            set(df.columns)
        ):

            candidate_metric_tables.append(
                (
                    file,
                    df
                )
            )

    except Exception as e:

        print(
            "Could not read:",
            file,
            "|",
            e
        )

print(
    "Usable metric tables:",
    len(candidate_metric_tables)
)

for file, df in candidate_metric_tables:

    print(
        os.path.basename(file),
        "->",
        df.shape
    )

Usable metric tables: 4
text_baseline_metrics.csv -> (7, 6)
image_baseline_metrics.csv -> (7, 6)
structured_baseline_metrics.csv -> (7, 6)
fusion_layer1_metrics.csv -> (7, 6)


In [9]:
# Inspect the available metric tables
for file, df in candidate_metric_tables:

    print(
        os.path.basename(file)
    )

    print(
        "================================"
    )

    display(
        df.head()
    )

text_baseline_metrics.csv


,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.823789,0.606742,0.915254,0.729730,0.946025
1,Support Devices,0.806167,0.660000,0.868421,0.750000,0.867811
2,Pleural Effusion,0.757709,0.507937,0.571429,0.537815,0.829470
3,Lung Opacity,0.889868,0.783784,0.865672,0.822695,0.952146
4,Atelectasis,0.845815,0.754386,0.671875,0.710744,0.927818


image_baseline_metrics.csv


,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.814978,0.666667,0.727273,0.695652,0.795502
1,Support Devices,0.647577,0.550000,0.819149,0.658120,0.729403
2,Pleural Effusion,0.585903,0.367647,0.862069,0.515464,0.797796
3,Lung Opacity,0.555066,0.375000,0.904762,0.530233,0.708188
4,Atelectasis,0.568282,0.351145,0.779661,0.484211,0.697437


structured_baseline_metrics.csv


,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.687225,0.459770,0.625000,0.529801,0.708493
1,Support Devices,0.651982,0.548387,0.579545,0.563536,0.684843
2,Pleural Effusion,0.590308,0.348624,0.633333,0.449704,0.668164
3,Lung Opacity,0.541850,0.305556,0.532258,0.388235,0.566520
4,Atelectasis,0.572687,0.320755,0.576271,0.412121,0.620510


fusion_layer1_metrics.csv


,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.911894,0.819672,0.847458,0.833333,0.918382
1,Support Devices,0.766520,0.623656,0.763158,0.686391,0.869990
2,Pleural Effusion,0.872247,0.736842,0.750000,0.743363,0.938074
3,Lung Opacity,0.947137,0.950820,0.865672,0.906250,0.982183
4,Atelectasis,0.885463,0.796875,0.796875,0.796875,0.950633


In [10]:
# Convert previous results into a common format
comparison_rows = []

for file, df in candidate_metric_tables:

    filename = (
        os.path.basename(file)
        .lower()
    )

    # Skip fusion because it is added separately
    if 'fusion' in filename:
        continue

    if 'text' in filename:

        model_name = (
            'Text'
        )

    elif 'image' in filename:

        model_name = (
            'Image'
        )

    elif (
        'structured' in filename
        or 'mlp' in filename
    ):

        model_name = (
            'Structured'
        )

    else:

        model_name = (
            os.path.basename(file)
            .replace('.csv', '')
        )

    comparison_rows.append({

        'Model': model_name,

        'Accuracy':
            df['accuracy'].mean(),

        'Precision':
            df['precision'].mean(),

        'Recall':
            df['recall'].mean(),

        'F1':
            df['f1'].mean(),

        'ROC-AUC':
            df['roc_auc'].mean()
    })

previous_comparison = pd.DataFrame(
    comparison_rows
)

# Add multimodal model
comparison_table = pd.concat(
    [
        previous_comparison,
        fusion_summary
    ],
    ignore_index=True
)

display(
    comparison_table
)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Text,0.829452,0.622734,0.806433,0.694337,0.912787
1,Image,0.609817,0.404681,0.828445,0.524583,0.724881
2,Structured,0.616740,0.363702,0.593705,0.444299,0.648917
3,Multimodal Fusion,0.896161,0.807109,0.807885,0.803855,0.942091


In [11]:
# Load saved feature representations

structured_file = (
    f'{processed_path}/'
    'structured_features_mlp_heldout.csv'
)

image_file = (
    f'{processed_path}/'
    'image_features_heldout.csv'
)

text_file = (
    f'{processed_path}/'
    'text_features_heldout.csv'
)

fusion_file = (
    f'{processed_path}/'
    'fusion_features_layer1.csv'
)

structured_features = pd.read_csv(
    structured_file
)

image_features = pd.read_csv(
    image_file
)

text_features = pd.read_csv(
    text_file
)

fusion_features = pd.read_csv(
    fusion_file
)

print(
    "Structured:",
    structured_features.shape
)

print(
    "Image:",
    image_features.shape
)

print(
    "Text:",
    text_features.shape
)

print(
    "Fusion:",
    fusion_features.shape
)

Structured: (687, 129)
Image: (687, 769)
Text: (687, 769)
Fusion: (1513, 129)


In [12]:
print("Structured columns:")
print(structured_features.columns.tolist())

print("\nImage columns:")
print(image_features.columns.tolist()[:10])

print("\nText columns:")
print(text_features.columns.tolist()[:10])

print("\nFusion columns:")
print(fusion_features.columns.tolist()[:10])

Structured columns:
['study_id', 'structured_feature_0', 'structured_feature_1', 'structured_feature_2', 'structured_feature_3', 'structured_feature_4', 'structured_feature_5', 'structured_feature_6', 'structured_feature_7', 'structured_feature_8', 'structured_feature_9', 'structured_feature_10', 'structured_feature_11', 'structured_feature_12', 'structured_feature_13', 'structured_feature_14', 'structured_feature_15', 'structured_feature_16', 'structured_feature_17', 'structured_feature_18', 'structured_feature_19', 'structured_feature_20', 'structured_feature_21', 'structured_feature_22', 'structured_feature_23', 'structured_feature_24', 'structured_feature_25', 'structured_feature_26', 'structured_feature_27', 'structured_feature_28', 'structured_feature_29', 'structured_feature_30', 'structured_feature_31', 'structured_feature_32', 'structured_feature_33', 'structured_feature_34', 'structured_feature_35', 'structured_feature_36', 'structured_feature_37', 'structured_feature_38', 's

In [13]:
# Load target labels
calibration_file = (
    f'{processed_path}/'
    'test_calibration_pool.csv'
)

calibration_df = pd.read_csv(
    calibration_file
)

TARGET = 'Pleural Effusion'

if TARGET not in calibration_df.columns:

    raise ValueError(
        f"Target '{TARGET}' was not found."
    )

y = (
    calibration_df[TARGET]
    .fillna(0)
    .replace(-1, 0)
    .astype(int)
)

print(
    "Target:",
    TARGET
)

print(
    "Target distribution:"
)

print(
    y.value_counts()
)

Target: Pleural Effusion
Target distribution:
Pleural Effusion
0    491
1    196
Name: count, dtype: int64


In [14]:
# Extract numeric feature matrix
def numeric_features(df):

    numeric_df = (
        df.select_dtypes(
            include=[np.number]
        )
        .copy()
    )

    return numeric_df

In [15]:
# Prepare feature matrices
X_structured = numeric_features(
    structured_features
)

X_image = numeric_features(
    image_features
)

X_text = numeric_features(
    text_features
)

X_fusion = numeric_features(
    fusion_features
)

print(
    "Structured numeric features:",
    X_structured.shape
)

print(
    "Image numeric features:",
    X_image.shape
)

print(
    "Text numeric features:",
    X_text.shape
)

print(
    "Fusion numeric features:",
    X_fusion.shape
)

Structured numeric features: (687, 129)
Image numeric features: (687, 769)
Text numeric features: (687, 769)
Fusion numeric features: (1513, 129)


In [16]:
# Align the available data lengths
n = min(
    len(y),
    len(X_structured),
    len(X_image),
    len(X_text),
    len(X_fusion)
)

y_test = y.iloc[
    :n
].to_numpy()

X_structured = X_structured.iloc[
    :n
].to_numpy()

X_image = X_image.iloc[
    :n
].to_numpy()

X_text = X_text.iloc[
    :n
].to_numpy()

X_fusion = X_fusion.iloc[
    :n
].to_numpy()

print(
    "Common sample size:",
    n
)

Common sample size: 687


In [17]:
# Lightweight models for statistical comparison

models = {

    'Structured': (
        X_structured
    ),

    'Text': (
        X_text
    ),

    'Image': (
        X_image
    ),

    'Multimodal Fusion': (
        X_fusion
    )
}

predictions = {}

probabilities = {}

for name, X in models.items():

    model = make_pipeline(

        StandardScaler(),

        LogisticRegression(
            max_iter=500,
            random_state=42
        )
    )

    model.fit(
        X,
        y_test
    )

    predictions[name] = (
        model.predict(X)
    )

    probabilities[name] = (
        model.predict_proba(X)[:, 1]
    )

    print(
        name,
        "completed"
    )

Structured completed
Text completed
Image completed
Multimodal Fusion completed


In [18]:
# Check ROC-AUC values
auc_results = []

for name in models.keys():

    auc = roc_auc_score(
        y_test,
        probabilities[name]
    )

    auc_results.append({

        'Model': name,

        'ROC-AUC': auc
    })

auc_table = pd.DataFrame(
    auc_results
)

display(
    auc_table
)

,Model,ROC-AUC
0,Structured,0.787211
1,Text,0.988622
2,Image,1.000000
3,Multimodal Fusion,0.708113


In [19]:
# McNemar tests

def run_mcnemar(
    y_true,
    pred_a,
    pred_b,
    name_a,
    name_b
):

    correct_a = (
        pred_a == y_true
    )

    correct_b = (
        pred_b == y_true
    )

    table = np.zeros(
        (2, 2),
        dtype=int
    )

    for a, b in zip(
        correct_a,
        correct_b
    ):

        table[
            int(a),
            int(b)
        ] += 1

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )

    return {

        'Comparison':
            f'{name_a} vs {name_b}',

        'Statistic':
            result.statistic,

        'p-value':
            result.pvalue
    }


mcnemar_results = []

for modality in [
    'Structured',
    'Text',
    'Image'
]:

    result = run_mcnemar(

        y_test,

        predictions[
            'Multimodal Fusion'
        ],

        predictions[
            modality
        ],

        'Multimodal Fusion',

        modality
    )

    mcnemar_results.append(
        result
    )


mcnemar_table = pd.DataFrame(
    mcnemar_results
)

display(
    mcnemar_table
)

,Comparison,Statistic,p-value
0,Multimodal Fusion vs Structured,1.730769,1.883127e-01
1,Multimodal Fusion vs Text,130.090323,3.915492e-30
2,Multimodal Fusion vs Image,182.005435,1.768339e-41


In [20]:
# DeLong ROC-AUC comparison

def compute_midrank(x):

    x = np.asarray(x)

    order = np.argsort(x)

    sorted_x = x[order]

    ranks = np.zeros(
        len(x),
        dtype=float
    )

    i = 0

    while i < len(x):

        j = i

        while (
            j < len(x)
            and sorted_x[j] == sorted_x[i]
        ):

            j += 1

        ranks[i:j] = (
            (i + j - 1) / 2.0
            + 1
        )

        i = j

    output = np.empty(
        len(x),
        dtype=float
    )

    output[order] = ranks

    return output


def delong_auc_variance(
    y_true,
    predictions
):

    y_true = np.asarray(
        y_true
    )

    predictions = np.asarray(
        predictions
    )

    positive = (
        predictions[
            y_true == 1
        ]
    )

    negative = (
        predictions[
            y_true == 0
        ]
    )

    m = len(positive)
    n = len(negative)

    if m == 0 or n == 0:

        return np.nan, np.nan

    combined = np.concatenate([
        positive,
        negative
    ])

    ranks = compute_midrank(
        combined
    )

    auc = (
        ranks[:m].sum()
        - m * (m + 1) / 2
    ) / (
        m * n
    )

    v01 = (
        ranks[:m]
        - np.arange(
            1,
            m + 1
        )
    ) / n

    v10 = (
        ranks[m:]
        - np.arange(
            m + 1,
            m + n + 1
        )
    ) / m

    sx = (
        np.var(
            v01,
            ddof=1
        ) / m
    )

    sy = (
        np.var(
            v10,
            ddof=1
        ) / n
    )

    variance = (
        sx + sy
    )

    return auc, variance

In [21]:
# DeLong tests

def delong_test(
    y_true,
    pred_a,
    pred_b
):

    auc_a, var_a = (
        delong_auc_variance(
            y_true,
            pred_a
        )
    )

    auc_b, var_b = (
        delong_auc_variance(
            y_true,
            pred_b
        )
    )

    se = np.sqrt(
        var_a + var_b
    )

    if se == 0:

        return (
            auc_a,
            auc_b,
            np.nan,
            np.nan
        )

    z = (
        auc_a - auc_b
    ) / se

    p_value = (
        2 * (
            1 - stats.norm.cdf(
                abs(z)
            )
        )
    )

    return (
        auc_a,
        auc_b,
        z,
        p_value
    )

In [22]:
# Compare multimodal ROC-AUC against each modality

delong_results = []

for modality in [
    'Structured',
    'Text',
    'Image'
]:

    auc_multi, auc_single, z, p = (
        delong_test(

            y_test,

            probabilities[
                'Multimodal Fusion'
            ],

            probabilities[
                modality
            ]
        )
    )

    delong_results.append({

        'Comparison':
            f'Multimodal Fusion vs {modality}',

        'Multimodal AUC':
            auc_multi,

        'Individual AUC':
            auc_single,

        'Z-statistic':
            z,

        'p-value':
            p
    })


delong_table = pd.DataFrame(
    delong_results
)

display(
    delong_table
)

,Comparison,Multimodal AUC,Individual AUC,Z-statistic,p-value
0,Multimodal Fusion vs Structured,0.708113,0.787211,-0.943438,0.345457
1,Multimodal Fusion vs Text,0.708113,0.988622,-3.554281,0.000379
2,Multimodal Fusion vs Image,0.708113,1.000000,-3.679062,0.000234


In [23]:
# Final statistical comparison table

statistical_summary = pd.DataFrame({

    'Comparison': [
        'Multimodal vs Structured',
        'Multimodal vs Text',
        'Multimodal vs Image'
    ],

    'McNemar p-value':
        mcnemar_table[
            'p-value'
        ].values,

    'DeLong p-value':
        delong_table[
            'p-value'
        ].values
})

display(
    statistical_summary
)

,Comparison,McNemar p-value,DeLong p-value
0,Multimodal vs Structured,1.883127e-01,0.345457
1,Multimodal vs Text,3.915492e-30,0.000379
2,Multimodal vs Image,1.768339e-41,0.000234
